# Information-Theoretic Wordle Solver (Strict Mode)

This notebook treats Wordle not as a guessing game, but as a **sequential inference problem** over a finite hypothesis space. At each round $ t $, we maintain a candidate set:

$$
\mathcal{S}_t \subseteq \mathcal{S}
$$

where $ \mathcal{S} $ is the set of all valid 5-letter English words.

A guess does not simply test a word - it **induces a partition** of the hypothesis space based on feedback. Each guess yields structured information that can be used to reduce uncertainty. We apply an **information-theoretic greedy strategy** based on **Shannon's decision tree principle** to minimize the size of the remaining search space.

We operate under **strict mode**: all guesses must be valid dictionary words (as in official Wordle rules). The inference strategy extends to unrestricted probing and larger alphabets, but this notebook focuses on the strict version first.


### Wordle as Sequential Inference

Let the target word $T$ be uniformly distributed over the dictionary $\mathcal{S}$. At round $t$, we maintain a **version space**:

$$
\mathcal{S}_1 = \mathcal{S}, \qquad
\mathcal{S}_{t+1} = \{\, w \in \mathcal{S}_t \mid F(g_t, w) = y_t \,\},
$$

where $g_t$ is our guess and $y_t = F(g_t, T)$ is the Wordle feedback vector returned by the oracle.

Each guess induces a **partition** of the current candidate space:

$$
\mathcal{S}_t = \biguplus_{y} C_y(g_t), \qquad
C_y(g_t) = \{\, w \in \mathcal{S}_t \mid F(g_t, w) = y \,\}.
$$

The objective is to shrink the hypothesis space $|\mathcal{S}_t|$ as efficiently as possible by selecting guesses that **maximize information gain**.


In [ ]:
# Load 5-letter Wordle dictionary
with open("words.txt","r") as f:
    WORD_LIST = [w.strip().lower() for w in f.readlines() if len(w.strip()) == 5]

len(WORD_LIST), WORD_LIST[:10]


### Wordle Feedback Function

Wordle provides structured feedback after each guess $g$ against the target $T$:

- **Green (G):** letter is correct and in the correct position  
- **Yellow (Y):** letter exists in the target but is in the wrong position  
- **Gray (\_):** letter does not appear in the target (after accounting for duplicates)

Formally, this feedback mechanism defines an oracle

$$
F: \Sigma^5 \times \Sigma^5 \rightarrow \{G, Y, \_\}^5,
$$

which maps a guess–target pair $(g, T)$ to a 5-symbol feedback string. This is the only information channel provided by the game, and it enables inference by iteratively reducing the hypothesis space.


In [ ]:
def wordle_feedback(guess, target):
    fb = ["_"] * 5
    t = list(target)

    # Green pass
    for i in range(5):
        if guess[i] == target[i]:
            fb[i] = "G"
            t[i] = None

    # Yellow pass
    for i in range(5):
        if fb[i] == "_" and guess[i] in t:
            fb[i] = "Y"
            t[t.index(guess[i])] = None

    return "".join(fb)

# Example
wordle_feedback("crane", "trace")


### Information-Theoretic Guess Selection

A guess $g$ partitions the version space $\mathcal{S}_t$ into feedback buckets:

$$
C_y(g) = \{\, w \in \mathcal{S}_t \mid F(g, w) = y \,\}.
$$

Each bucket corresponds to a possible feedback pattern $y$. Since the true target must belong to exactly one of these buckets, making a guess is equivalent to selecting a partition of $\mathcal{S}_t$.

We select guesses that minimize the **expected size** of the next candidate set:

$$
\mathbb{E}\big[\,|\mathcal{S}_{t+1}| \mid g\,\big]
= \sum_{y} \frac{|C_y(g)|}{|\mathcal{S}_t|} \cdot |C_y(g)|
= \frac{1}{|\mathcal{S}_t|} \sum_{y} |C_y(g)|^2.
$$

Since $|\mathcal{S}_t|$ is constant at step $t$, we minimize the score

$$
J(g) = \sum_{y} |C_y(g)|^2.
$$

This implements **Shannon's greedy decision tree strategy**: at each step, choose the query that yields the **maximum information gain** by producing the most balanced partition of the hypothesis space.


In [ ]:
def score_guess(guess, candidates):
    buckets = {}
    for w in candidates:
        fb = wordle_feedback(guess, w)
        buckets[fb] = buckets.get(fb, 0) + 1
    return sum(v*v for v in buckets.values()), max(buckets.values())  # (J, W)


### Best Guess Selection

To select the optimal guess at step $t$, we use a **lexicographic decision rule**:

$$
g_t = \arg\min_{g \in \mathcal{G}} \Big( J(g),\; W(g),\; \mathbf{1}\{g \notin \mathcal{S}_t\} \Big),
$$

where:

- $J(g)$ measures **expected uncertainty after guessing $g$** by minimizing the expected size of the next version space.
- $W(g) = \max_{y} |C_y(g)|$ controls the **worst-case partition**, preventing highly unbalanced splits.
- The indicator term $\mathbf{1}\{g \notin \mathcal{S}_t\}$ ensures that, in the case of ties, we prefer guesses still present in the current candidate set.

This rule balances **information-theoretic optimality** (via $J(g)$) and **search robustness** (via $W(g)$), producing a greedy but effective inference strategy.


In [ ]:
def best_guess(candidates, all_words):
    best = None
    best_score = (10**18, 10**18)
    for g in all_words:
        J, W = score_guess(g, candidates)
        if (J, W) < best_score or ((J, W) == best_score and g in candidates and (best not in candidates if best else True)):
            best_score = (J, W)
            best = g
    return best


### Feedback Visualization

To support gameplay interaction, we present feedback using Wordle color symbols.


In [ ]:
def show_feedback(fb):
    return "".join("🟩" if c=="G" else "🟨" if c=="Y" else "⬛" for c in fb)


### Full Solver: Human-AI Collaboration

The AI acts as an **information guide**, suggesting optimal moves based on the current posterior candidate set $ \mathcal{S}_t $. The AI:
1. Recomputes best guesses at every turn.
2. Always suggests from the full dictionary.
3. Never crashes if the user plays differently.
4. Continues inference normally from feedback.


In [ ]:
import random

def play_with_ai():
    target = random.choice(WORD_LIST)
    candidates = WORD_LIST.copy()
    print("New Wordle game started (strict mode).\n")

    # AI suggests before user even starts
    suggestion = best_guess(candidates, WORD_LIST)
    print(f"Best next suggestion: {suggestion.upper()}")
    print(f"Words remaining: {len(candidates)}\n")

    while True:
        guess = input("Enter guess: ").strip().lower()
        if len(guess) != 5 or guess not in WORD_LIST:
            print("Invalid guess. Must be a valid 5-letter word.")
            continue

        fb = wordle_feedback(guess, target)
        print("Feedback:", show_feedback(fb))

        if fb == "GGGGG":
            print(f"✅ Correct! The word was: {target.upper()}")
            break

        # Shrink candidate space
        candidates = [w for w in candidates if wordle_feedback(guess, w) == fb]

        suggestion = best_guess(candidates, WORD_LIST)
        print(f"Best next suggestion: {suggestion.upper()}")
        print(f"Words remaining: {len(candidates)}\n")


In [ ]:
play_with_ai()